# V9.2 — Partie 2 : Normalisation, Data Quality, FIELD_CONFIDENCE et DQ_SCORE

Cette partie ne relance **jamais Qwen**. Elle travaille uniquement sur les JSON `DOM_EXTRACTION_V1`.

Objectifs :
- normalisation Python conservatrice et traçable ;
- propagation des signaux réglementaires de la Partie 1 V9.2 et des dossiers V8.1 migrés ;
- cross-check entre documents ;
- contrôles métier déterministes ;
- `FIELD_CONFIDENCE` explicable par champ ;
- `DQ_SCORE` explicable par dossier ;
- `AUTO_OK / REVIEW / BLOCKED` pour une validation humaine par exception.

**Important :** `FIELD_CONFIDENCE` et `DQ_SCORE` sont des indices internes de Data Quality. Ils ne représentent pas une probabilité statistique et ne remplacent pas le futur chantier **Qwen Confidence**, qui sera traité séparément avant mise en production.


### Ajustements métier V2

- `CTS_SAP_ID`, `TTR_NUMERO_MANUSCRIT`, toutes les données `PTR_*`,
  `CTS_NUMERO_SS_PAYS_ORIGINE` et `CTS_NUMERO_SS_ALGERIE` sont **INFO_ONLY** :
  elles restent visibles mais ne pénalisent pas le DQ_SCORE.
- `CTS_SALAIRE_NET_ANCIEN` est non applicable pour un `NOUVEAU_CONTRAT`
  et devient pertinent pour une `AUGMENTATION`.
- `DOM_MONTANT_TOTAL_DOMICILIE` est non applicable pour une `AUGMENTATION`.
- `TYPE_DOSSIER` est dérivé : `NOUVEAU_CONTRAT`, `AUGMENTATION` ou `A_DETERMINER`.
- `CTS_DATE_AUGMENTATION` est dérivée de `CTS_DATE_DOCUMENT` pour une augmentation.
- les montants avec une décimale explicite sont acceptés :
  `47 719 857,6 → 47719857.60`.


## 1. Imports et configuration


In [ ]:
import hashlib
import json
import math
import re
from collections import defaultdict
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd

SCHEMA_VERSION = 'DOM_EXTRACTION_V1'
EXPECTED_FIELD_SCHEMA_HASH = 'da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e'
POSTPROCESS_VERSION = 'DOM_V9_2_PART2_DQ_CONFIDENCE_V2'
NORMALIZATION_VERSION = 'NORM_V9_2_CONSERVATIVE_V2'
MAX_DOSSIERS = None   # mettre 10 pour un test

OUTPUT_ROOT = Path('/mnt/data/domiciliations_v9')
RAW_JSON_DIR = OUTPUT_ROOT / '01_extraction_raw' / 'json_dossiers'
POST_ROOT = OUTPUT_ROOT / '02_postprocessing'
PROCESSED_JSON_DIR = POST_ROOT / 'processed_json'
VALIDATION_XLSX = POST_ROOT / 'validation_domiciliations_v9.xlsx'
DOSSIERS_CSV = POST_ROOT / 'dossiers_a_valider.csv'
FIELDS_CSV = POST_ROOT / 'champs_detail.csv'
ANOMALIES_CSV = POST_ROOT / 'anomalies.csv'
RETRY_CSV = POST_ROOT / 'vlm_retry_requests.csv'
DQ_SUMMARY_CSV = POST_ROOT / 'dq_summary.csv'
FIELD_CONFIDENCE_CSV = POST_ROOT / 'field_confidence.csv'

POST_ROOT.mkdir(parents=True,exist_ok=True)
PROCESSED_JSON_DIR.mkdir(parents=True,exist_ok=True)

print('RAW :',RAW_JSON_DIR)
print('Post:',POST_ROOT)


## 2. Schéma — mêmes 99 champs V8.1 / V9


In [ ]:
FIELD_SCHEMA = {'ENGAGEMENT_DOMICILIATION': ['DOM_NOM_RAISON_SOCIAL_CLIENT', 'DOM_COMPTE_LOCAL', 'DOM_ADRESSE_CLIENT', 'DOM_AGENCE_DOMICILIATAIRE', 'DOM_NUMERO_CONTRAT', 'DOM_DUREE_CONTRAT_MOIS', 'DOM_DATE_DEBUT_CONTRAT', 'DOM_DATE_FIN_CONTRAT', 'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR', 'DOM_ADRESSE_EMPLOYEUR', 'DOM_SALAIRE_NET_MENSUEL', 'DOM_PART_TRANSFERABLE', 'DOM_TAUX_TRANSFERABLE', 'DOM_MONTANT_TOTAL_DOMICILIE', 'DOM_DATE_SIGNATURE'], 'CONTRAT_TRAVAIL': ['CTR_REFERENCE_DOCUMENT', 'CTR_TYPE', 'CTR_EMPLOYEUR', 'CTR_ACTIVITE_EMPLOYEUR', 'CTR_DUREE_MOIS', 'CTR_DATE_DEBUT_CONTRAT', 'CTR_POSTE', 'CTR_NOM_PRENOM_TRAVAILLEUR', 'CTR_PERE_NOM_PRENOM', 'CTR_MERE_NOM_PRENOM', 'CTR_NATIONALITE', 'CTR_DATE_NAISSANCE', 'CTR_LIEU_PAYS_NAISSANCE', 'CTR_ADRESSE_ALGERIE', 'CTR_QUALIFICATION', 'CTR_NUMERO_PERMIS_TRAVAIL', 'CTR_DATE_DELIVRANCE_PERMIS', 'CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS', 'CTR_SALAIRE_BRUT', 'CTR_SALAIRE_NET', 'CTR_AFFILIATION_SS', 'CTR_NUMERO_EMPLOYEUR', 'CTR_DATE_SIGNATURE', 'CTR_REFERENCE_DOMICILIATION', 'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTR_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTR_CACHET_EMPLOYEUR_PRESENT'], 'CONTRAT_SPECIFIQUE': ['CTS_REFERENCE_DOCUMENT', 'CTS_SAP_ID', 'CTS_EMPLOYEUR', 'CTS_ACTIVITE_EMPLOYEUR', 'CTS_DUREE_MOIS', 'CTS_DATE_DEBUT_CONTRAT', 'CTS_POSTE', 'CTS_NOM_PRENOM_TRAVAILLEUR', 'CTS_PERE_NOM_PRENOM', 'CTS_MERE_NOM_PRENOM', 'CTS_NATIONALITE', 'CTS_DATE_NAISSANCE', 'CTS_LIEU_PAYS_NAISSANCE', 'CTS_ADRESSE_ALGERIE', 'CTS_QUALIFICATION', 'CTS_NUMERO_PERMIS_TRAVAIL', 'CTS_DATE_DELIVRANCE_PERMIS', 'CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS', 'CTS_LIGNE_SALAIRE_BRUTE', 'CTS_SALAIRE_NET', 'CTS_SALAIRE_NET_ANCIEN', 'CTS_MENTION_AU_LIEU_DE_PRESENTE', 'CTS_PART_TRANSFERABLE', 'CTS_PART_PAYABLE_DZD', 'CTS_NUMERO_SS_PAYS_ORIGINE', 'CTS_NUMERO_SS_ALGERIE', 'CTS_DATE_DOCUMENT', 'CTS_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTS_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTS_CACHET_EMPLOYEUR_PRESENT', 'CTS_VISA_INSPECTION_TRAVAIL_PRESENT'], 'TITRE_TRAVAIL': ['TTR_NUMERO_PERMIS', 'TTR_NUMERO_MANUSCRIT', 'TTR_POSTE', 'TTR_DUREE', 'TTR_DATE_DEBUT', 'TTR_DATE_FIN', 'TTR_LIEU_TRAVAIL', 'TTR_EMPLOYEUR', 'TTR_ADRESSE_EMPLOYEUR', 'TTR_FAIT_A', 'TTR_DATE_DELIVRANCE', 'TTR_NOM', 'TTR_PRENOM', 'TTR_DATE_NAISSANCE', 'TTR_LIEU_NAISSANCE', 'TTR_PAYS', 'TTR_NATIONALITE', 'TTR_QUALIFICATION', 'TTR_DATE_ENTREE_ALGERIE', 'TTR_PHOTO_PRESENTE', 'TTR_CACHET_PRESENT'], 'PERMIS_TRAVAIL_COUVERTURE': ['PTR_NUMERO_SERIE', 'PTR_WILAYA', 'PTR_CACHET_DIRECTION_EMPLOI_PRESENT']}

ALL_FIELDS=[f for fields in FIELD_SCHEMA.values() for f in fields]
assert len(ALL_FIELDS)==99 and len(set(ALL_FIELDS))==99
schema_hash=hashlib.sha256(json.dumps(FIELD_SCHEMA,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
assert schema_hash==EXPECTED_FIELD_SCHEMA_HASH
print('✅ Schéma 99 champs vérifié |',schema_hash[:16]+'…')


## 3. Typage des champs — configuration évolutive


In [ ]:
AMOUNT_FIELDS={
    'DOM_SALAIRE_NET_MENSUEL','DOM_PART_TRANSFERABLE','DOM_MONTANT_TOTAL_DOMICILIE',
    'CTR_SALAIRE_BRUT','CTR_SALAIRE_NET','CTS_SALAIRE_NET','CTS_SALAIRE_NET_ANCIEN',
    'CTS_PART_TRANSFERABLE','CTS_PART_PAYABLE_DZD',
}
PERCENT_FIELDS={'DOM_TAUX_TRANSFERABLE'}
INTEGER_FIELDS={'DOM_DUREE_CONTRAT_MOIS','CTR_DUREE_MOIS','CTS_DUREE_MOIS'}
BOOLEAN_FIELDS={
    'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE','CTR_SIGNATURE_EMPLOYEUR_PRESENTE','CTR_CACHET_EMPLOYEUR_PRESENT',
    'CTS_MENTION_AU_LIEU_DE_PRESENTE','CTS_SIGNATURE_TRAVAILLEUR_PRESENTE','CTS_SIGNATURE_EMPLOYEUR_PRESENTE',
    'CTS_CACHET_EMPLOYEUR_PRESENT','CTS_VISA_INSPECTION_TRAVAIL_PRESENT',
    'TTR_PHOTO_PRESENTE','TTR_CACHET_PRESENT','PTR_CACHET_DIRECTION_EMPLOI_PRESENT',
}
DATE_FIELDS={f for f in ALL_FIELDS if '_DATE_' in f or f.startswith('TTR_DATE_') or f in {'DOM_DATE_SIGNATURE','CTS_DATE_DOCUMENT'}}
REFERENCE_FIELDS={
    'DOM_COMPTE_LOCAL','DOM_NUMERO_CONTRAT','CTR_REFERENCE_DOCUMENT','CTR_NUMERO_PERMIS_TRAVAIL',
    'CTR_REFERENCE_DOMICILIATION','CTS_REFERENCE_DOCUMENT','CTS_SAP_ID','CTS_NUMERO_PERMIS_TRAVAIL',
    'TTR_NUMERO_PERMIS','TTR_NUMERO_MANUSCRIT','PTR_NUMERO_SERIE',
}

FIELD_TYPES={}
for f in ALL_FIELDS:
    if f in AMOUNT_FIELDS: FIELD_TYPES[f]='amount'
    elif f in PERCENT_FIELDS: FIELD_TYPES[f]='percentage'
    elif f in INTEGER_FIELDS: FIELD_TYPES[f]='integer'
    elif f in BOOLEAN_FIELDS: FIELD_TYPES[f]='boolean'
    elif f in DATE_FIELDS: FIELD_TYPES[f]='date'
    elif f in REFERENCE_FIELDS: FIELD_TYPES[f]='reference'
    else: FIELD_TYPES[f]='text'

print(pd.Series(FIELD_TYPES).value_counts())


## 4. Normalisation traçable


In [ ]:
NULL_TEXTS={'','NULL','NONE','N/A','NA','NÉANT','NEANT','ILLISIBLE','NON LISIBLE'}

def result(raw,normalized,status,rule,message=None):
    return {'raw':raw,'normalized':normalized,'status':status,'rule':rule,
            'changed':(normalized != raw),'message':message}


def normalize_amount_trace(raw):
    """
    Normalisation conservatrice des montants.
    Principe réglementaire :
      - ne jamais modifier/inventer un chiffre ;
      - AUTO_OK uniquement si la structure des séparateurs est déterministe ;
      - REVIEW dès qu'une interprétation économique reste plausible.
    """
    if raw is None:
        return result(raw,None,'MISSING','AMOUNT_NULL')

    if isinstance(raw,(int,float,Decimal)) and not isinstance(raw,bool):
        try:
            return result(raw,round(float(raw),2),'RAW_OK','AMOUNT_NUMERIC')
        except Exception:
            pass

    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS:
        return result(raw,None,'MISSING','AMOUNT_NULL_TEXT')

    # Retirer seulement les libellés de devise connus.
    t=s.upper().replace('DZD','').replace('DA','').replace('EUR','').replace('€','').strip()
    neg=t.startswith('-')
    t=t.lstrip('+-').strip()

    # Caractères non autorisés : aucune correction OCR de type O->0 / B->8.
    if not re.fullmatch(r"[0-9., '\u2019]+", t or ''):
        return result(raw,None,'REVIEW','AMOUNT_NON_NUMERIC',
                      'caractère non numérique ambigu')

    # Les séparateurs espace/apostrophe sont acceptés comme milliers
    # seulement si leurs groupes sont structurellement cohérents.
    if re.search(r"[ '\u2019]", t):
        chunks=[x for x in re.split(r"[ '\u2019]+",t) if x]
        # Cas "454 835,67" / "454 835.67" : groupes milliers + décimales explicites.
        if len(chunks) >= 2:
            last=chunks[-1]
            # Si le dernier bloc isolé contient 1 ou 2 chiffres sans . ou ,
            # ex. "454 835 67", on refuse de deviner qu'il s'agit de décimales.
            if re.fullmatch(r'\d{1,2}', last):
                return result(raw,None,'REVIEW','AMOUNT_SPACE_DECIMAL_AMBIGUOUS',
                              'dernier groupe espace/apostrophe ambigu')
            # Tous les groupes intermédiaires doivent être des milliers.
            for ch in chunks[1:-1]:
                if not re.fullmatch(r'\d{3}', ch):
                    return result(raw,None,'REVIEW','AMOUNT_SPACE_GROUPING_AMBIGUOUS')
        t=re.sub(r"[ '\u2019]",'',t)

    if not re.fullmatch(r'[0-9.,]+',t or ''):
        return result(raw,None,'REVIEW','AMOUNT_PARSE_FAILED')

    dots=t.count('.')
    commas=t.count(',')
    dec_sep=None
    rule=''

    if dots and commas:
        last_dot=t.rfind('.')
        last_comma=t.rfind(',')
        candidate='.' if last_dot>last_comma else ','
        tail=t.split(candidate)[-1]
        if len(tail)==2:
            dec_sep=candidate
            rule='AMOUNT_MIXED_LAST_2_DECIMALS'
        elif len(tail)==1:
            dec_sep=candidate
            rule='AMOUNT_MIXED_LAST_1_DECIMAL_PAD_ZERO'
        else:
            return result(raw,None,'REVIEW','AMOUNT_MIXED_AMBIGUOUS')

    elif dots>1 or commas>1:
        sep='.' if dots else ','
        groups=t.split(sep)
        tail=groups[-1]

        # Exemple validé : 454.835.67 -> 454835.67
        if len(tail)==2 and all(g.isdigit() for g in groups):
            dec_sep=sep
            rule='AMOUNT_MULTI_GROUP_FINAL_2_DECIMALS'
        elif len(tail)==1 and all(g.isdigit() for g in groups) and all(len(g)==3 for g in groups[1:-1]):
            dec_sep=sep
            rule='AMOUNT_MULTI_GROUP_FINAL_1_DECIMAL_PAD_ZERO'
        elif all(len(g)==3 for g in groups[1:]):
            dec_sep=None
            rule='AMOUNT_MULTI_THOUSANDS'
        else:
            return result(raw,None,'REVIEW','AMOUNT_MULTI_AMBIGUOUS')

    elif dots==1 or commas==1:
        sep='.' if dots else ','
        left,right=t.split(sep)
        if len(right)==2:
            dec_sep=sep
            rule='AMOUNT_SINGLE_2_DECIMALS'
        elif len(right)==1:
            # Le séparateur décimal est explicite : 47719857,6 = 47719857.60.
            # Aucun chiffre économique n'est inventé ; le zéro final est
            # uniquement une représentation à 2 décimales.
            dec_sep=sep
            rule='AMOUNT_SINGLE_1_DECIMAL_PAD_ZERO'
        elif len(right)==3:
            return result(raw,None,'REVIEW','AMOUNT_SINGLE_3DIGITS_AMBIGUOUS')
        else:
            return result(raw,None,'REVIEW','AMOUNT_SINGLE_AMBIGUOUS')
    else:
        rule='AMOUNT_INTEGER'

    if dec_sep:
        pos=t.rfind(dec_sep)
        int_part=re.sub(r'[.,]','',t[:pos])
        dec=t[pos+1:]
        canonical=int_part+'.'+dec
    else:
        canonical=re.sub(r'[.,]','',t)

    if neg:
        canonical='-'+canonical

    try:
        val=round(float(Decimal(canonical)),2)
    except (InvalidOperation,ValueError):
        return result(raw,None,'REVIEW','AMOUNT_PARSE_FAILED')

    return result(raw,val,'AUTO_OK' if str(raw)!=str(val) else 'RAW_OK',rule)


DATE_FORMATS=['%d/%m/%Y','%d-%m-%Y','%d.%m.%Y','%Y-%m-%d','%Y/%m/%d','%Y.%m.%d']
def normalize_date_trace(raw):
    if raw is None: return result(raw,None,'MISSING','DATE_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','DATE_NULL_TEXT')
    s=re.sub(r'\s+','',s)
    for fmt in DATE_FORMATS:
        try:
            dt=datetime.strptime(s,fmt)
            out=dt.strftime('%d/%m/%Y')
            return result(raw,out,'RAW_OK' if s==out else 'AUTO_OK','DATE_'+fmt.replace('%',''))
        except ValueError:
            pass
    return result(raw,None,'REVIEW','DATE_UNPARSEABLE')


def normalize_integer_trace(raw):
    if raw is None: return result(raw,None,'MISSING','INTEGER_NULL')
    if isinstance(raw,int) and not isinstance(raw,bool): return result(raw,raw,'RAW_OK','INTEGER_NUMERIC')
    s=str(raw).strip(); m=re.fullmatch(r'\s*(\d+)\s*(?:mois)?\s*',s,flags=re.I)
    if not m: return result(raw,None,'REVIEW','INTEGER_AMBIGUOUS')
    v=int(m.group(1)); return result(raw,v,'RAW_OK' if str(v)==s else 'AUTO_OK','INTEGER_EXTRACT')


def normalize_percentage_trace(raw):
    if raw is None: return result(raw,None,'MISSING','PERCENT_NULL')
    s=str(raw).strip().replace('\u00a0',' ')
    has_pct='%' in s
    s=s.replace('%','').replace(' ','').replace(',','.')
    if not re.fullmatch(r'[+-]?\d+(?:\.\d+)?',s): return result(raw,None,'REVIEW','PERCENT_AMBIGUOUS')
    v=float(s)
    if not has_pct and 0 < v <= 1:
        v*=100; rule='PERCENT_FRACTION_TO_PERCENT'
    else: rule='PERCENT_DIRECT'
    if not (0 <= v <= 100): return result(raw,None,'REVIEW','PERCENT_OUT_OF_RANGE')
    v=round(v,2); return result(raw,v,'AUTO_OK' if str(raw).strip()!=str(v) else 'RAW_OK',rule)


def normalize_boolean_trace(raw):
    if raw is None: return result(raw,None,'MISSING','BOOL_NULL')
    if isinstance(raw,bool): return result(raw,raw,'RAW_OK','BOOL_NATIVE')
    s=str(raw).strip().upper()
    if s in {'TRUE','VRAI','OUI','YES','1'}: return result(raw,True,'AUTO_OK','BOOL_TRUE_TEXT')
    if s in {'FALSE','FAUX','NON','NO','0'}: return result(raw,False,'AUTO_OK','BOOL_FALSE_TEXT')
    return result(raw,None,'REVIEW','BOOL_AMBIGUOUS')


def normalize_reference_trace(raw):
    if raw is None: return result(raw,None,'MISSING','REF_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','REF_NULL_TEXT')
    # Conservateur : espaces périphériques et autour de / - uniquement.
    out=re.sub(r'\s*([/\-])\s*',r'\1',re.sub(r'\s+',' ',s)).strip()
    return result(raw,out,'AUTO_OK' if out!=s else 'RAW_OK','REF_SPACING_ONLY')


def normalize_text_trace(raw):
    if raw is None: return result(raw,None,'MISSING','TEXT_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','TEXT_NULL_TEXT')
    # Pas de correction orthographique / casse.
    return result(raw,s,'AUTO_OK' if s!=raw else 'RAW_OK','TEXT_TRIM_ONLY')


def normalize_field_trace(field,raw):
    typ=FIELD_TYPES.get(field,'text')
    return {
        'amount':normalize_amount_trace,
        'date':normalize_date_trace,
        'integer':normalize_integer_trace,
        'percentage':normalize_percentage_trace,
        'boolean':normalize_boolean_trace,
        'reference':normalize_reference_trace,
        'text':normalize_text_trace,
    }[typ](raw)

# Tests de non-régression des montants.
assert normalize_amount_trace('454.835.67')['normalized']==454835.67
assert normalize_amount_trace('23.340.43')['normalized']==23340.43
assert normalize_amount_trace('23,340.43')['normalized']==23340.43
assert normalize_amount_trace('23.340,43')['normalized']==23340.43
assert normalize_amount_trace('23 340,43')['normalized']==23340.43
assert normalize_amount_trace('23,340')['status']=='REVIEW'
assert normalize_amount_trace('454 835 67')['status']=='REVIEW'
assert normalize_amount_trace('454.835.6')['normalized']==454835.60
assert normalize_amount_trace('47 719 857,6')['normalized']==47719857.60
assert normalize_amount_trace('47.719.857,6')['normalized']==47719857.60
assert normalize_amount_trace('454 835 67')['status']=='REVIEW'
assert normalize_amount_trace('454.83O.67')['status']=='REVIEW'
print('✅ Montants testés : 454.835.67 -> 454835.67 | 47 719 857,6 -> 47719857.60')


## 5. Contrôles croisés — sans appel Qwen


In [ ]:
CROSS_DOCUMENT_GROUPS=[
    {'name':'SALAIRE_NET','kind':'amount','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_SALAIRE_NET_MENSUEL','CONTRAT_TRAVAIL':'CTR_SALAIRE_NET','CONTRAT_SPECIFIQUE':'CTS_SALAIRE_NET'}},
    {'name':'PART_TRANSFERABLE','kind':'amount','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_PART_TRANSFERABLE','CONTRAT_SPECIFIQUE':'CTS_PART_TRANSFERABLE'}},
    {'name':'DATE_DEBUT_CONTRAT','kind':'date','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_DATE_DEBUT_CONTRAT','CONTRAT_TRAVAIL':'CTR_DATE_DEBUT_CONTRAT','CONTRAT_SPECIFIQUE':'CTS_DATE_DEBUT_CONTRAT'}},
    {'name':'NUMERO_PERMIS','kind':'reference','fields':{
        'CONTRAT_TRAVAIL':'CTR_NUMERO_PERMIS_TRAVAIL','CONTRAT_SPECIFIQUE':'CTS_NUMERO_PERMIS_TRAVAIL','TITRE_TRAVAIL':'TTR_NUMERO_PERMIS'}},
    {'name':'DATE_DEBUT_PERMIS','kind':'date','fields':{
        'CONTRAT_TRAVAIL':'CTR_DATE_DEBUT_VALIDITE_PERMIS','CONTRAT_SPECIFIQUE':'CTS_DATE_DEBUT_VALIDITE_PERMIS','TITRE_TRAVAIL':'TTR_DATE_DEBUT'}},
    {'name':'DATE_FIN_PERMIS','kind':'date','fields':{
        'CONTRAT_TRAVAIL':'CTR_DATE_FIN_VALIDITE_PERMIS','CONTRAT_SPECIFIQUE':'CTS_DATE_FIN_VALIDITE_PERMIS','TITRE_TRAVAIL':'TTR_DATE_FIN'}},
]

def comparable(v): return v not in (None,'')
def same_value(kind,a,b):
    if not comparable(a) or not comparable(b): return True
    if kind=='amount': return abs(float(a)-float(b))<=0.01
    return str(a)==str(b)


In [ ]:

# =====================================================================
# FIELD_CONFIDENCE + DQ_SCORE + APPLICABILITE METIER
# =====================================================================
# Ces scores sont des INDICES INTERNES EXPLICABLES, pas des probabilités
# statistiques et pas un "taux de confiance Qwen".

CRITICAL_FIELDS = {
    'DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT',
    'DOM_SALAIRE_NET_MENSUEL','DOM_PART_TRANSFERABLE',
    'CTR_NUMERO_PERMIS_TRAVAIL','CTR_DATE_DEBUT_VALIDITE_PERMIS',
    'CTR_DATE_FIN_VALIDITE_PERMIS','CTR_SALAIRE_NET',
    'CTS_NUMERO_PERMIS_TRAVAIL','CTS_DATE_DEBUT_VALIDITE_PERMIS',
    'CTS_DATE_FIN_VALIDITE_PERMIS','CTS_SALAIRE_NET',
    'TTR_NUMERO_PERMIS','TTR_DATE_DEBUT','TTR_DATE_FIN',
    'TTR_NOM','TTR_PRENOM'
}

# Champs explicitement considérés comme informatifs / non déterminants
# pour le DQ_SCORE et la décision AUTO_OK / REVIEW / BLOCKED.
#
# Noms exacts du schéma :
# - CTS_SAP_ID (le besoin métier a été exprimé comme CTS_SAP_IP)
# - CTS_NUMERO_SS_ALGERIE
ALWAYS_INFO_ONLY_FIELDS = {
    'CTS_SAP_ID',
    'TTR_NUMERO_MANUSCRIT',
    'PTR_NUMERO_SERIE',
    'PTR_WILAYA',
    'PTR_CACHET_DIRECTION_EMPLOI_PRESENT',
    'CTS_NUMERO_SS_PAYS_ORIGINE',
    'CTS_NUMERO_SS_ALGERIE',
}

NON_PENALIZING_QUALITY_FLAGS = {
    'MIGRATED_FROM_V8_1',
}

BLOCKING_ANOMALIES = {
    'CLASSIFICATION_REVIEW_REQUIRED',
    'EXTRACTION_JSON_VIDE',
    'CRITICAL_FIELD_MISSING',
    'CROSS_DOCUMENT_CONFLICT_CRITICAL',
    'INVALID_DATE_ORDER',
    'AUGMENTATION_DATE_MISSING',
}

DQ_WEIGHTS = {
    'critical_completeness': 25,
    'cross_document': 30,
    'normalization': 15,
    'business_validity': 15,
    'extraction_quality': 10,
    'classification_quality': 5,
}
assert sum(DQ_WEIGHTS.values()) == 100


def _meaningful(v):
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip().upper() not in NULL_TEXTS
    return True


def _first_value(records, field):
    for r in records:
        v=(r.get('normalized_data') or {}).get(field)
        if comparable(v):
            return v
    return None


def derive_business_context(processed_records):
    """
    Déduit la nature du dossier sans modifier les 99 champs RAW.

    AUGMENTATION si :
      - CTS_SALAIRE_NET_ANCIEN est renseigné, ou
      - CTS_MENTION_AU_LIEU_DE_PRESENTE == True.

    Sinon, si un CONTRAT_SPECIFIQUE existe :
      NOUVEAU_CONTRAT.

    CTS_DATE_AUGMENTATION est une donnée DERIVEE :
      = CTS_DATE_DOCUMENT uniquement pour une AUGMENTATION.
    """
    cts_records=[r for r in processed_records if r.get('doc_type')=='CONTRAT_SPECIFIQUE']
    has_cts=bool(cts_records)

    old_salary=None
    mention_au_lieu=False
    cts_date_document=None

    for r in cts_records:
        nd=r.get('normalized_data') or {}
        if old_salary is None and comparable(nd.get('CTS_SALAIRE_NET_ANCIEN')):
            old_salary=nd.get('CTS_SALAIRE_NET_ANCIEN')
        if nd.get('CTS_MENTION_AU_LIEU_DE_PRESENTE') is True:
            mention_au_lieu=True
        if cts_date_document is None and comparable(nd.get('CTS_DATE_DOCUMENT')):
            cts_date_document=nd.get('CTS_DATE_DOCUMENT')

    evidence=[]
    if comparable(old_salary):
        evidence.append('CTS_SALAIRE_NET_ANCIEN_PRESENT')
    if mention_au_lieu:
        evidence.append('CTS_MENTION_AU_LIEU_DE_PRESENTE_TRUE')

    if evidence:
        type_dossier='AUGMENTATION'
    elif has_cts:
        type_dossier='NOUVEAU_CONTRAT'
        evidence.append('CTS_SANS_INDICATEUR_AUGMENTATION')
    else:
        type_dossier='A_DETERMINER'
        evidence.append('ABSENCE_CONTRAT_SPECIFIQUE')

    date_aug=cts_date_document if type_dossier=='AUGMENTATION' else None

    return {
        'TYPE_DOSSIER':type_dossier,
        'TYPE_DOSSIER_MOTIF':' | '.join(evidence),
        'CTS_DATE_AUGMENTATION':date_aug,
    }


def field_policy(field, business_context):
    """
    Retourne (APPLICABILITE, DQ_IMPORTANCE).

    - INFO_ONLY : visible dans les données mais ne crée pas d'anomalie DQ.
    - NON_APPLICABLE : champ non attendu pour ce type de dossier.
    - CRITICAL / STANDARD : champ pris en compte dans la validation.
    """
    typ=(business_context or {}).get('TYPE_DOSSIER')

    if field in ALWAYS_INFO_ONLY_FIELDS:
        return 'INFO_ONLY','INFO_ONLY'

    if field=='CTS_SALAIRE_NET_ANCIEN' and typ=='NOUVEAU_CONTRAT':
        return 'NON_APPLICABLE','INFO_ONLY'

    if field=='DOM_MONTANT_TOTAL_DOMICILIE' and typ=='AUGMENTATION':
        return 'NON_APPLICABLE','INFO_ONLY'

    if field in CRITICAL_FIELDS:
        return 'APPLICABLE','CRITICAL'

    return 'APPLICABLE','STANDARD'


def is_dq_relevant_field(field, business_context):
    _, importance=field_policy(field,business_context)
    return importance!='INFO_ONLY'


def is_info_only_document(doc_type, business_context):
    fields=FIELD_SCHEMA.get(doc_type,[])
    return bool(fields) and all(not is_dq_relevant_field(f,business_context) for f in fields)


def relevant_critical_missing(rec, business_context):
    return [
        f for f in (rec.get('critical_fields_missing') or [])
        if is_dq_relevant_field(f,business_context)
    ]


def penalizing_quality_flags(rec):
    return [
        str(x) for x in (rec.get('quality_flags') or [])
        if str(x) not in NON_PENALIZING_QUALITY_FLAGS
    ]


def parse_date_safe(v):
    if not v:
        return None
    try:
        return datetime.strptime(str(v), '%d/%m/%Y')
    except Exception:
        return None


def page_regulatory_flags(rec, business_context=None):
    flags=[]

    # Si tout le document est purement informatif (PTR), ses anomalies
    # n'impactent pas la décision réglementaire du dossier.
    info_doc=is_info_only_document(rec.get('doc_type'), business_context)

    if rec.get('classification_review_required') is True and not info_doc:
        flags.append('CLASSIFICATION_REVIEW_REQUIRED')

    status=str(rec.get('extraction_status') or '').upper()
    if not info_doc:
        if status in {'JSON_VIDE','VIDE','FAILED','ECHEC'}:
            flags.append('EXTRACTION_JSON_VIDE')
        elif status in {'PARTIELLE','PARTIAL'}:
            flags.append('EXTRACTION_PARTIELLE')

        if relevant_critical_missing(rec,business_context):
            flags.append('CRITICAL_FIELD_MISSING')

    for qf in penalizing_quality_flags(rec):
        flags.append(qf)

    return list(dict.fromkeys(flags))


def field_evidence(processed_records, field):
    ev=[]
    for r in processed_records:
        nd=r.get('normalized_data') or {}
        tr=r.get('normalization_trace') or {}
        if field in nd:
            ev.append({
                'doc_type':r.get('doc_type'),
                'page':r.get('page_num'),
                'value':nd.get(field),
                'raw':(r.get('raw_data') or {}).get(field),
                'trace':tr.get(field) or {},
                'classification_review_required':bool(r.get('classification_review_required')),
                'quality_flags':penalizing_quality_flags(r),
                'extraction_status':r.get('extraction_status'),
            })
    return ev


def calculate_field_confidence(processed_records, business_context):
    rows=[]
    for field in ALL_FIELDS:
        applicability,importance=field_policy(field,business_context)
        ev=field_evidence(processed_records,field)
        present=[e for e in ev if comparable(e['value'])]

        if importance=='INFO_ONLY':
            rows.append({
                'CHAMP':field,
                'APPLICABILITE':applicability,
                'DQ_IMPORTANCE':importance,
                'FIELD_CONFIDENCE':None,
                'FIELD_CONFIDENCE_LEVEL':'INFO_ONLY',
                'FIELD_CONFIDENCE_REASON':'HORS_SCORE_DQ',
                'NB_OCCURRENCES':len(present),
            })
            continue

        if not present:
            score=0
            level='MISSING'
            reasons=['AUCUNE_VALEUR_NORMALISEE']
        else:
            score=70
            reasons=['VALEUR_PRESENTE']

            statuses=[(e['trace'] or {}).get('status') for e in present]
            if any(s=='REVIEW' for s in statuses):
                score-=35
                reasons.append('NORMALISATION_REVIEW')
            elif all(s in {'RAW_OK','AUTO_OK'} for s in statuses):
                score+=10
                reasons.append('NORMALISATION_DETERMINISTE')

            if any(e['classification_review_required'] for e in present):
                score-=30
                reasons.append('CLASSIFICATION_REVIEW')

            if any(str(e['extraction_status'] or '').upper() in
                   {'PARTIELLE','PARTIAL','JSON_VIDE','FAILED','ECHEC'} for e in present):
                score-=20
                reasons.append('EXTRACTION_NON_COMPLETE')

            if any(e['quality_flags'] for e in present):
                score-=5
                reasons.append('QUALITY_FLAG_PRESENT')

            vals=[e['value'] for e in present]
            if len(vals)>=2:
                kind=FIELD_TYPES.get(field,'text')
                if all(same_value(kind,vals[0],x) for x in vals[1:]):
                    score+=20
                    reasons.append(f'CONCORDANCE_{len(vals)}_LECTURES')
                else:
                    score-=35
                    reasons.append('DIVERGENCE_INTER_OCCURRENCES')

            score=max(0,min(100,int(round(score))))
            level='HIGH' if score>=90 else ('MEDIUM' if score>=70 else 'LOW')

        rows.append({
            'CHAMP':field,
            'APPLICABILITE':applicability,
            'DQ_IMPORTANCE':importance,
            'FIELD_CONFIDENCE':score,
            'FIELD_CONFIDENCE_LEVEL':level,
            'FIELD_CONFIDENCE_REASON':' | '.join(reasons),
            'NB_OCCURRENCES':len(present),
        })
    return rows


def add_business_controls(processed, anomalies, business_context):
    records=processed['page_records']

    values={}
    for r in records:
        for f,v in (r.get('normalized_data') or {}).items():
            if comparable(v) and f not in values:
                values[f]=v

    date_pairs=[
        ('DOM_DATE_DEBUT_CONTRAT','DOM_DATE_FIN_CONTRAT','CONTRAT'),
        ('CTR_DATE_DEBUT_VALIDITE_PERMIS','CTR_DATE_FIN_VALIDITE_PERMIS','PERMIS_CTR'),
        ('CTS_DATE_DEBUT_VALIDITE_PERMIS','CTS_DATE_FIN_VALIDITE_PERMIS','PERMIS_CTS'),
        ('TTR_DATE_DEBUT','TTR_DATE_FIN','PERMIS_TTR'),
    ]
    for f1,f2,label in date_pairs:
        d1=parse_date_safe(values.get(f1)); d2=parse_date_safe(values.get(f2))
        if d1 and d2 and d2 < d1:
            anomalies.append({
                'FICHIER':processed['source_file'],'PAGE':None,'TYPE_DOCUMENT':'MULTI',
                'TYPE_ANOMALIE':'INVALID_DATE_ORDER','SEVERITE':'BLOQUANT','CHAMP':label,
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':f'{values.get(f1)} > {values.get(f2)}',
                'MOTIF':'date fin antérieure à date début'
            })

    # Pour une augmentation, la date du document CTS pilote le mois
    # à partir duquel le nouveau salaire sera applicable.
    if business_context.get('TYPE_DOSSIER')=='AUGMENTATION':
        if not business_context.get('CTS_DATE_AUGMENTATION'):
            anomalies.append({
                'FICHIER':processed['source_file'],'PAGE':None,'TYPE_DOCUMENT':'CONTRAT_SPECIFIQUE',
                'TYPE_ANOMALIE':'AUGMENTATION_DATE_MISSING','SEVERITE':'BLOQUANT',
                'CHAMP':'CTS_DATE_AUGMENTATION',
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'augmentation détectée mais CTS_DATE_DOCUMENT absente/invalide'
            })


def calculate_dq_summary(processed, field_conf_rows):
    records=processed['page_records']
    anomalies=processed.get('anomalies') or []
    business_context=processed.get('business_context') or {}

    critical_scores=[]
    field_map={x['CHAMP']:x for x in field_conf_rows}
    for f in CRITICAL_FIELDS:
        if not is_dq_relevant_field(f,business_context):
            continue
        owner=None
        for dt,fields in FIELD_SCHEMA.items():
            if f in fields:
                owner=dt
                break
        if owner and any(r.get('doc_type')==owner for r in records):
            score=field_map[f].get('FIELD_CONFIDENCE')
            critical_scores.append(1 if score not in (None,0) else 0)
    critical_completeness=(sum(critical_scores)/len(critical_scores)) if critical_scores else 0

    cross_conflicts=[a for a in anomalies if a.get('TYPE_ANOMALIE')=='CROSS_DOCUMENT_CONFLICT']
    cross_checks_possible=0
    for g in CROSS_DOCUMENT_GROUPS:
        vals=[]
        for dt,f in g['fields'].items():
            for r in records:
                if r.get('doc_type')==dt:
                    v=(r.get('normalized_data') or {}).get(f)
                    if comparable(v):
                        vals.append(v)
        if len(vals)>=2:
            cross_checks_possible+=1

    cross_document=0.60 if cross_checks_possible==0 else max(
        0,1-(len(cross_conflicts)/cross_checks_possible)
    )

    traces=[]
    for r in records:
        for field,tr in (r.get('normalization_trace') or {}).items():
            if is_dq_relevant_field(field,business_context):
                traces.append(tr)
    nonmissing=[t for t in traces if t.get('status')!='MISSING']
    normalization=(
        sum(t.get('status') in {'RAW_OK','AUTO_OK'} for t in nonmissing)/len(nonmissing)
        if nonmissing else 0
    )

    business_bad=sum(
        a.get('TYPE_ANOMALIE') in {'INVALID_DATE_ORDER','AUGMENTATION_DATE_MISSING'}
        for a in anomalies
    )
    business_validity=1.0 if business_bad==0 else 0.0

    page_flags=[page_regulatory_flags(r,business_context) for r in records]
    severe_pages=sum(
        any(x in {'EXTRACTION_JSON_VIDE','EXTRACTION_PARTIELLE','CRITICAL_FIELD_MISSING'} for x in fl)
        for fl in page_flags
    )
    extraction_quality=max(0,1-(severe_pages/max(1,len(records))))

    class_review=sum('CLASSIFICATION_REVIEW_REQUIRED' in fl for fl in page_flags)
    classification_quality=max(0,1-(class_review/max(1,len(records))))

    components={
        'critical_completeness':critical_completeness,
        'cross_document':cross_document,
        'normalization':normalization,
        'business_validity':business_validity,
        'extraction_quality':extraction_quality,
        'classification_quality':classification_quality,
    }
    dq_score=round(sum(components[k]*DQ_WEIGHTS[k] for k in DQ_WEIGHTS),1)

    hard_reasons=[]
    for r in records:
        hard_reasons.extend(
            x for x in page_regulatory_flags(r,business_context)
            if x in BLOCKING_ANOMALIES
        )

    for a in anomalies:
        typ=a.get('TYPE_ANOMALIE')
        if typ in {'INVALID_DATE_ORDER','AUGMENTATION_DATE_MISSING'}:
            hard_reasons.append(typ)
        if typ=='CROSS_DOCUMENT_CONFLICT':
            if a.get('CHAMP') in {
                'SALAIRE_NET','NUMERO_PERMIS',
                'DATE_DEBUT_PERMIS','DATE_FIN_PERMIS'
            }:
                hard_reasons.append('CROSS_DOCUMENT_CONFLICT_CRITICAL')

    hard_reasons=list(dict.fromkeys(hard_reasons))

    if hard_reasons:
        validation='BLOCKED'
    elif dq_score >= 90 and not any(
        a.get('TYPE_ANOMALIE')=='FORMAT_REVIEW' for a in anomalies
    ):
        validation='AUTO_OK'
    else:
        validation='REVIEW'

    level='HIGH' if dq_score>=90 else ('MEDIUM' if dq_score>=70 else 'LOW')

    return {
        'TYPE_DOSSIER':business_context.get('TYPE_DOSSIER'),
        'CTS_DATE_AUGMENTATION':business_context.get('CTS_DATE_AUGMENTATION'),
        'DQ_SCORE':dq_score,
        'DQ_LEVEL':level,
        'VALIDATION_AUTO':validation,
        'HARD_BLOCK':bool(hard_reasons),
        'HARD_BLOCK_REASONS':' | '.join(hard_reasons),
        'SCORE_CRITICAL_COMPLETENESS':round(critical_completeness*100,1),
        'SCORE_CROSS_DOCUMENT':round(cross_document*100,1),
        'SCORE_NORMALIZATION':round(normalization*100,1),
        'SCORE_BUSINESS_VALIDITY':round(business_validity*100,1),
        'SCORE_EXTRACTION_QUALITY':round(extraction_quality*100,1),
        'SCORE_CLASSIFICATION_QUALITY':round(classification_quality*100,1),
    }

print('✅ DQ V2 chargé : applicabilité métier + nouveau contrat/augmentation + champs info-only')


## 6. Traitement d’un JSON RAW


In [ ]:

def validate_raw_contract(d):
    return (d.get('schema_version')==SCHEMA_VERSION and
            d.get('field_schema_hash')==EXPECTED_FIELD_SCHEMA_HASH and
            isinstance(d.get('page_records'),list))


def process_raw_dossier(d):
    if not validate_raw_contract(d):
        raise ValueError(f"Contrat RAW incompatible : {d.get('source_file')}")

    source=d['source_file']
    processed_records=[]
    anomalies=[]
    retry=[]

    # --------------------------------------------------------------
    # PASSAGE 1 : normaliser, sans décider encore si un REVIEW
    # est métier-important ou seulement informatif.
    # --------------------------------------------------------------
    for rec in d['page_records']:
        dt=rec.get('doc_type')
        raw=dict(rec.get('raw_data') or {})
        normalized={}
        trace={}

        for field in FIELD_SCHEMA.get(dt,[]):
            tr=normalize_field_trace(field,raw.get(field))
            trace[field]=tr
            normalized[field]=tr['normalized']

        p=dict(rec)
        p['normalized_data']=normalized
        p['normalization_trace']=trace
        processed_records.append(p)

    # La nature du dossier est dérivée APRES normalisation.
    business_context=derive_business_context(processed_records)

    # --------------------------------------------------------------
    # PASSAGE 2 : appliquer l'applicabilité métier.
    # --------------------------------------------------------------
    field_rows=[]

    for rec in processed_records:
        dt=rec.get('doc_type')
        page=rec.get('page_num')
        raw=rec.get('raw_data') or {}
        trace=rec.get('normalization_trace') or {}

        # Signaux réglementaires, filtrés selon l'importance métier.
        flags=page_regulatory_flags(rec,business_context)
        relevant_missing=relevant_critical_missing(rec,business_context)

        if 'CLASSIFICATION_REVIEW_REQUIRED' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'CLASSIFICATION_REVIEW_REQUIRED','SEVERITE':'BLOQUANT',
                'CHAMP':None,'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'classification page à revoir'
            })

        if 'EXTRACTION_JSON_VIDE' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'EXTRACTION_JSON_VIDE','SEVERITE':'BLOQUANT',
                'CHAMP':None,'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'extraction vide/échouée'
            })

        if 'EXTRACTION_PARTIELLE' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'EXTRACTION_PARTIELLE','SEVERITE':'REVIEW',
                'CHAMP':None,'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'extraction partielle'
            })

        if 'CRITICAL_FIELD_MISSING' in flags:
            anomalies.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                'TYPE_ANOMALIE':'CRITICAL_FIELD_MISSING','SEVERITE':'BLOQUANT',
                'CHAMP':' | '.join(relevant_missing),
                'VALEUR_RAW':None,'VALEUR_NORMALISEE':None,
                'MOTIF':'champ critique métier manquant'
            })

        for field in FIELD_SCHEMA.get(dt,[]):
            tr=trace[field]
            applicability,importance=field_policy(field,business_context)

            field_rows.append({
                'FICHIER':source,
                'TYPE_DOSSIER':business_context.get('TYPE_DOSSIER'),
                'PAGE':page,
                'TYPE_DOCUMENT':dt,
                'CHAMP':field,
                'TYPE_CHAMP':FIELD_TYPES[field],
                'APPLICABILITE':applicability,
                'DQ_IMPORTANCE':importance,
                'RAW':tr['raw'],
                'NORMALIZED':tr['normalized'],
                'STATUS':tr['status'],
                'RULE':tr['rule'],
                'CHANGED':tr['changed'],
                'MESSAGE':tr.get('message'),
            })

            # Un REVIEW sur un champ INFO_ONLY / NON_APPLICABLE reste visible
            # dans CHAMPS_DETAIL mais ne génère ni anomalie ni retry.
            if tr['status']=='REVIEW' and importance!='INFO_ONLY':
                anomalies.append({
                    'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                    'TYPE_ANOMALIE':'FORMAT_REVIEW','SEVERITE':'REVIEW',
                    'CHAMP':field,'VALEUR_RAW':tr['raw'],
                    'VALEUR_NORMALISEE':tr['normalized'],
                    'MOTIF':tr['rule']
                })
                retry.append({
                    'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,
                    'CHAMP':field,'VALEUR_RAW':tr['raw'],
                    'MOTIF':'FORMAT_AMBIGU','RULE':tr['rule']
                })

    # --------------------------------------------------------------
    # CROSS-CHECK sur valeurs normalisées valides.
    # --------------------------------------------------------------
    by_type=defaultdict(list)
    for r in processed_records:
        by_type[r.get('doc_type')].append(r)

    for group in CROSS_DOCUMENT_GROUPS:
        vals=[]
        for dt,field in group['fields'].items():
            for r in by_type.get(dt,[]):
                v=(r.get('normalized_data') or {}).get(field)
                if comparable(v):
                    vals.append((
                        dt,field,r.get('page_num'),v,
                        (r.get('raw_data') or {}).get(field)
                    ))

        if len(vals)>=2:
            base=vals[0][3]
            if any(not same_value(group['kind'],base,x[3]) for x in vals[1:]):
                detail=' | '.join(
                    f'{dt}.{field}@p{p}={v}'
                    for dt,field,p,v,raw in vals
                )
                anomalies.append({
                    'FICHIER':source,'PAGE':None,'TYPE_DOCUMENT':'MULTI',
                    'TYPE_ANOMALIE':'CROSS_DOCUMENT_CONFLICT',
                    'SEVERITE':'BLOQUANT' if group['name'] in {
                        'SALAIRE_NET','NUMERO_PERMIS','DATE_DEBUT_PERMIS','DATE_FIN_PERMIS'
                    } else 'REVIEW',
                    'CHAMP':group['name'],'VALEUR_RAW':None,
                    'VALEUR_NORMALISEE':detail,
                    'MOTIF':'valeurs divergentes entre documents'
                })
                for dt,field,p,v,raw in vals:
                    retry.append({
                        'FICHIER':source,'PAGE':p,'TYPE_DOCUMENT':dt,
                        'CHAMP':field,'VALEUR_RAW':raw,
                        'MOTIF':'CROSS_DOCUMENT_CONFLICT',
                        'RULE':group['name']
                    })

    # --------------------------------------------------------------
    # Contrôles métier + scores.
    # --------------------------------------------------------------
    temp_processed={
        'source_file':source,
        'page_records':processed_records,
        'business_context':business_context,
    }
    add_business_controls(temp_processed,anomalies,business_context)
    field_confidence=calculate_field_confidence(
        processed_records,business_context
    )

    out={
        'schema_version':SCHEMA_VERSION,
        'source_file':source,
        'source_sha256':d.get('source_sha256'),
        'source_pipeline_version':d.get('pipeline_version'),
        'postprocess_version':POSTPROCESS_VERSION,
        'normalization_version':NORMALIZATION_VERSION,
        'processed_at':datetime.now().isoformat(timespec='seconds'),
        'business_context':business_context,
        'page_records':processed_records,
        'anomalies':anomalies,
        'retry_requests':retry,
        'field_confidence':field_confidence,
        'source_stats':d.get('stats') or {},
    }

    dq_summary=calculate_dq_summary(out,field_confidence)
    out['dq_summary']=dq_summary

    return out,field_rows,anomalies,retry,field_confidence,dq_summary


def first_non_null(*values):
    for v in values:
        if v not in (None,''):
            return v
    return None


def consolidate_for_validation(processed):
    records=processed['page_records']
    ctx=processed.get('business_context') or {}

    # Les colonnes métier dérivées sont placées au début du fichier Excel.
    row={
        'FICHIER':processed['source_file'],
        'TYPE_DOSSIER':ctx.get('TYPE_DOSSIER'),
        'TYPE_DOSSIER_MOTIF':ctx.get('TYPE_DOSSIER_MOTIF'),
        'CTS_DATE_AUGMENTATION':ctx.get('CTS_DATE_AUGMENTATION'),
    }

    row['NB_PAGES']=len(set(
        r.get('page_num') for r in records
        if r.get('page_num') is not None
    ))
    row['TYPES_DOCUMENTS']=' | '.join(dict.fromkeys(
        str(r.get('doc_type')) for r in records
    ))

    sources={}
    for r in records:
        dt=r.get('doc_type')
        page=r.get('page_num')
        nd=r.get('normalized_data') or {}
        for f,v in nd.items():
            if f not in row or row.get(f) in (None,''):
                row[f]=v
                sources[f]=f'{dt}@p{page}'

    for f in ALL_FIELDS:
        row.setdefault(f,None)

    row['NOM_TRAVAILLEUR_REFERENCE']=first_non_null(
        row.get('CTR_NOM_PRENOM_TRAVAILLEUR'),
        row.get('CTS_NOM_PRENOM_TRAVAILLEUR'),
        ' '.join(
            x for x in [
                str(row.get('TTR_NOM') or '').strip(),
                str(row.get('TTR_PRENOM') or '').strip()
            ] if x
        ) or None
    )

    # PTR_NUMERO_SERIE n'est plus utilisé comme fallback :
    # toutes les données PTR sont informatives.
    row['NUMERO_PERMIS_REFERENCE']=first_non_null(
        row.get('TTR_NUMERO_PERMIS'),
        row.get('CTR_NUMERO_PERMIS_TRAVAIL'),
        row.get('CTS_NUMERO_PERMIS_TRAVAIL')
    )

    row['DATE_DEBUT_CONTRAT_REFERENCE']=first_non_null(
        row.get('DOM_DATE_DEBUT_CONTRAT'),
        row.get('CTR_DATE_DEBUT_CONTRAT'),
        row.get('CTS_DATE_DEBUT_CONTRAT')
    )
    row['DATE_FIN_CONTRAT_REFERENCE']=first_non_null(
        row.get('DOM_DATE_FIN_CONTRAT')
    )
    row['SALAIRE_REFERENCE']=first_non_null(
        row.get('CTS_SALAIRE_NET'),
        row.get('CTR_SALAIRE_NET'),
        row.get('DOM_SALAIRE_NET_MENSUEL')
    )
    row['PART_TRANSFERABLE_REFERENCE']=first_non_null(
        row.get('DOM_PART_TRANSFERABLE'),
        row.get('CTS_PART_TRANSFERABLE')
    )

    row['NB_ANOMALIES']=len(processed.get('anomalies') or [])

    dq=processed.get('dq_summary') or {}
    for k,v in dq.items():
        # TYPE_DOSSIER et CTS_DATE_AUGMENTATION sont déjà au début.
        if k not in {'TYPE_DOSSIER','CTS_DATE_AUGMENTATION'}:
            row[k]=v

    row['NB_RETRY_REQUESTS']=len(processed.get('retry_requests') or [])
    row['STATUT_VALIDATION']=row.get('VALIDATION_AUTO','REVIEW')
    row['COMMENTAIRE_VALIDATION']=None

    return row,sources


## 7. Exécution Partie 2 et classeur de validation


In [ ]:
json_files=sorted(RAW_JSON_DIR.glob('*.json'))
if MAX_DOSSIERS is not None:
    json_files=json_files[:int(MAX_DOSSIERS)]
print('JSON RAW sélectionnés :',len(json_files))

all_dossiers=[]; all_fields=[]; all_anomalies=[]; all_retry=[]; all_docs=[]; errors=[]
all_field_confidence=[]; all_dq_summary=[]
for i,p in enumerate(json_files,1):
    try:
        raw=json.loads(p.read_text(encoding='utf-8'))
        processed,field_rows,anomalies,retry,field_confidence,dq_summary=process_raw_dossier(raw)
        row,sources=consolidate_for_validation(processed)
        all_dossiers.append(row); all_fields.extend(field_rows); all_anomalies.extend(anomalies); all_retry.extend(retry)
        all_field_confidence.extend([{'FICHIER':processed['source_file'], **x} for x in field_confidence])
        all_dq_summary.append({
            'FICHIER':processed['source_file'],
            'TYPE_DOSSIER':(processed.get('business_context') or {}).get('TYPE_DOSSIER'),
            'CTS_DATE_AUGMENTATION':(processed.get('business_context') or {}).get('CTS_DATE_AUGMENTATION'),
            **dq_summary,
            'NB_ANOMALIES':len(anomalies)
        })
        for r in processed['page_records']:
            all_docs.append({'FICHIER':processed['source_file'],'PAGE':r.get('page_num'),'TYPE_DOCUMENT':r.get('doc_type'),
                             'STATUT_EXTRACTION':r.get('extraction_status'),'TAUX_REMPLISSAGE':r.get('extraction_taux_remplissage'),
                             'CRITICAL_FIELDS_MISSING_RAW':' | '.join(r.get('critical_fields_missing') or []),
                             'CRITICAL_FIELDS_MISSING_DQ':' | '.join(
                                 relevant_critical_missing(r,processed.get('business_context') or {})
                             ),
                             'STRATEGIES':' > '.join(s.get('nom','') for s in (r.get('extraction_strategies') or [])),
                             'TOKENS_IN':r.get('extraction_tokens_in'),'TOKENS_OUT':r.get('extraction_tokens_out')})
        outpath=PROCESSED_JSON_DIR/p.name
        outpath.write_text(json.dumps(processed,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
        print(f'[{i}/{len(json_files)}] ✅ {p.name} | anomalies={len(anomalies)}')
    except Exception as exc:
        errors.append({'FICHIER':p.name,'ERREUR':repr(exc)})
        print(f'[{i}/{len(json_files)}] ❌ {p.name}: {exc}')

DF_DOSSIERS=pd.DataFrame(all_dossiers)
DF_FIELDS=pd.DataFrame(all_fields)
DF_ANOMALIES=pd.DataFrame(all_anomalies)
DF_RETRY=pd.DataFrame(all_retry).drop_duplicates() if all_retry else pd.DataFrame(columns=['FICHIER','PAGE','TYPE_DOCUMENT','CHAMP','VALEUR_RAW','MOTIF','RULE'])
DF_DOCS=pd.DataFrame(all_docs)
DF_ERRORS=pd.DataFrame(errors)
DF_FIELD_CONFIDENCE=pd.DataFrame(all_field_confidence)
DF_DQ_SUMMARY=pd.DataFrame(all_dq_summary)

DF_DOSSIERS.to_csv(DOSSIERS_CSV,index=False,encoding='utf-8-sig')
DF_FIELDS.to_csv(FIELDS_CSV,index=False,encoding='utf-8-sig')
DF_ANOMALIES.to_csv(ANOMALIES_CSV,index=False,encoding='utf-8-sig')
DF_RETRY.to_csv(RETRY_CSV,index=False,encoding='utf-8-sig')
DF_FIELD_CONFIDENCE.to_csv(FIELD_CONFIDENCE_CSV,index=False,encoding='utf-8-sig')
DF_DQ_SUMMARY.to_csv(DQ_SUMMARY_CSV,index=False,encoding='utf-8-sig')

with pd.ExcelWriter(VALIDATION_XLSX,engine='openpyxl') as writer:
    DF_DQ_SUMMARY.to_excel(writer,sheet_name='DQ_DASHBOARD',index=False)
    DF_DOSSIERS.to_excel(writer,sheet_name='DOSSIERS_A_VALIDER',index=False)
    DF_FIELDS.to_excel(writer,sheet_name='CHAMPS_DETAIL',index=False)
    DF_FIELD_CONFIDENCE.to_excel(writer,sheet_name='FIELD_CONFIDENCE',index=False)
    DF_ANOMALIES.to_excel(writer,sheet_name='ANOMALIES',index=False)
    DF_RETRY.to_excel(writer,sheet_name='VLM_RETRY_REQUESTS',index=False)
    DF_DOCS.to_excel(writer,sheet_name='DOCUMENTS',index=False)
    DF_ERRORS.to_excel(writer,sheet_name='ERREURS',index=False)
    pd.DataFrame([
        {'PARAMETRE':'schema_version','VALEUR':SCHEMA_VERSION},
        {'PARAMETRE':'field_schema_hash','VALEUR':EXPECTED_FIELD_SCHEMA_HASH},
        {'PARAMETRE':'postprocess_version','VALEUR':POSTPROCESS_VERSION},
        {'PARAMETRE':'normalization_version','VALEUR':NORMALIZATION_VERSION},
        {'PARAMETRE':'dq_score_note','VALEUR':'Indice interne explicable 0-100; ce n est pas une probabilité ni un taux de confiance Qwen'},
        {'PARAMETRE':'qwen_confidence','VALEUR':'NON INTEGRE - chantier pré-production séparé'},
        {'PARAMETRE':'type_dossier_rule','VALEUR':'AUGMENTATION si CTS_SALAIRE_NET_ANCIEN présent ou CTS_MENTION_AU_LIEU_DE_PRESENTE=True; sinon NOUVEAU_CONTRAT si CTS présent'},
        {'PARAMETRE':'augmentation_date_rule','VALEUR':'CTS_DATE_AUGMENTATION = CTS_DATE_DOCUMENT uniquement pour AUGMENTATION'},
    ]).to_excel(writer,sheet_name='PARAMETRES',index=False)

    # Affichage Excel de tous les montants avec 2 décimales.
    ws=writer.book['DOSSIERS_A_VALIDER']
    amount_cols=set(AMOUNT_FIELDS) | {'SALAIRE_REFERENCE','PART_TRANSFERABLE_REFERENCE'}
    headers={cell.value:cell.column for cell in ws[1]}
    for col_name in amount_cols:
        col_idx=headers.get(col_name)
        if col_idx:
            for row_idx in range(2,ws.max_row+1):
                cell=ws.cell(row=row_idx,column=col_idx)
                if isinstance(cell.value,(int,float)):
                    cell.number_format='0.00'

    # Dans CHAMPS_DETAIL, NORMALIZED est formaté à 2 décimales
    # seulement lorsque TYPE_CHAMP == amount.
    ws=writer.book['CHAMPS_DETAIL']
    headers={cell.value:cell.column for cell in ws[1]}
    type_col=headers.get('TYPE_CHAMP')
    norm_col=headers.get('NORMALIZED')
    if type_col and norm_col:
        for row_idx in range(2,ws.max_row+1):
            if ws.cell(row=row_idx,column=type_col).value=='amount':
                cell=ws.cell(row=row_idx,column=norm_col)
                if isinstance(cell.value,(int,float)):
                    cell.number_format='0.00'

print('\n✅ Classeur validation :',VALIDATION_XLSX)
print('✅ Dossiers CSV       :',DOSSIERS_CSV)
print('✅ Champs détail      :',FIELDS_CSV)
print('✅ Anomalies          :',ANOMALIES_CSV)
print('✅ Retry requests     :',RETRY_CSV)
print('✅ DQ summary         :',DQ_SUMMARY_CSV)
print('✅ Field confidence   :',FIELD_CONFIDENCE_CSV)


## 8. Étape suivante — après validation des 10 dossiers

Ne pas construire automatiquement la base finale ni recalculer `PLANNING_TL` tant que les règles de normalisation et de consolidation ne sont pas validées sur les 10 dossiers.

Une fois validé, la suite de **cette même Partie 2** ajoutera :
1. `DOMICILIATIONS` — 1 ligne par domiciliation ;
2. `DOM_VERSIONS` — initiale, augmentation 1, augmentation 2, etc. ;
3. `PLANNING_TL` — 1 ligne par mois avec `EXECUTE`, `STATUT`, `DATE_EXECUTION`, `MONTANT_EXECUTE`, `MOTIF_REJET`, `COMMENTAIRE`.

Le futur programme de migration des anciens JSON V8.x aura une seule fonction : les convertir en `DOM_EXTRACTION_V1`, puis ils passeront dans cette Partie 2 exactement comme les nouveaux dossiers.
